In [1]:
!nvidia-smi

Sat Sep 26 19:45:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers datasets accelerate trl peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 29.2 MB/s eta 0:00:00


In [3]:
model_name = "Qwen/Qwen2.5-0.5B"

In [4]:
# pre-trained tokenizer to convert words into tokens
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokens = tokenizer("Hello world how are you?")
print(tokens)
print(tokenizer.decode(tokens['input_ids']))


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

{'input_ids': [9707, 1879, 1246, 525, 498, 30], 'attention_mask': [1, 1, 1, 1, 1, 1]}
Hello world how are you?


In [5]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, device_map='auto')

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [6]:
from datasets import load_dataset
dataset = load_dataset("timdettmers/openassistant-guanaco")

README.md:   0%|          | 0.00/395 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


openassistant_best_replies_train.jsonl: reconstructing file:   0%|          |  0.00B / 20.9MB            

openassistant_best_replies_train.jsonl: downloading bytes:           |  0.00B            

openassistant_best_replies_eval.jsonl:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9846 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/518 [00:00<?, ? examples/s]

In [7]:
print(dataset)
print(dataset['train'][0])
import random
for i in random.sample(range(len(dataset['train'])), 3):
    print(dataset['train'][i]['text'][:300])
    print('---')

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 9846
    })
    test: Dataset({
        features: ['text'],
        num_rows: 518
    })
})
{'text': '### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant: "Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.\n\nRecent research has identified potential monopsonies in industries such as retail and fast food, where a few large 

In [9]:
def generate(prompt, max_new_token=80):
  input = tokenizer(prompt, return_tensors='pt').to(model.device)
  out = model.generate(**input, max_new_tokens=max_new_token)
  return tokenizer.decode(out[0],skip_special_tokens=True)

test_prompt = "### Human: What is the capital of France? ### Assistant:"
generate(test_prompt)

'### Human: What is the capital of France? ### Assistant: The capital of France is Paris.'

In [13]:
#lets try from dataset
sample_texts = [dataset['train'][i]['text'] for i in range(0,5)]
test_prompts = [t.split('### Assistant: ')[0] for t in sample_texts]

for p in test_prompts:
  print("Prompt:", p[:150])
  print("Model:", generate(p))
  print("------")


Prompt: ### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monop
Model: ### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research. 

### Machine: Sure, I can help you with that. Here's a short introduction about the relevance of the term "monopsony" in economics:

Monopsony is a market structure in which a single buyer or seller in a market has the power to set prices and restrict the supply of goods or services. In the context of the labour market, a monopsony can
------
Prompt: ### Human: ¿CUales son las etapas del desarrollo y en qué consisten según Piaget?
Model: ### Human: ¿CUales son las etapas del desarrollo y en qué consisten según Piaget? ¿Qué se conoce como el "nuevo concepto" y cómo se desarrolla en el desarrollo de los niños?

Assistant:

In [ ]:
from trl import SFTConfig, SFTTrainer
training_args = SFTConfig(
                  output_dir = "./qwen-sft-training",
                  num_train_epochs = 1,
                  per_device_train_batch_size = 4,
                  logging_steps = 4,
                  save_steps = 50,
                  eval_strategy="steps",
                  eval_steps=5)

trainer = SFTTrainer(model=model,
                     args = training_args,
                     train_dataset = dataset['train'],
                     eval_dataset = dataset['test'])

trainer.train()


Adding EOS to train dataset:   0%|          | 0/9846 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9846 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/9846 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/9846 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/9846 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
5,2.015086,1.842841,1.926146,5253.000000,0.596106
10,1.720038,1.827472,1.868462,11576.000000,0.598037
15,1.905146,1.819550,1.875002,19125.000000,0.598753
20,1.903272,1.816217,1.871986,27443.000000,0.599035
25,1.705623,1.815146,1.878074,33158.000000,0.599402
30,1.756316,1.813721,1.903044,40342.000000,0.599419
35,1.883738,1.812794,1.875238,47759.000000,0.599371
40,1.766226,1.810450,1.861293,53340.000000,0.599712
45,1.753538,1.809978,1.860698,61557.000000,0.600127
50,2.102418,1.809248,1.893122,67153.000000,0.599086


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]